# CAP4D Static Avatar for Unity (Local Jupyter / Docker)
This notebook runs tracking + generation + avatar fitting and exports a static Unity-friendly 3DGS `.ply`.

When launched in the provided Docker container, it uses the mounted repo directly and exposes JupyterLab in your browser.


In [ ]:
import os
from pathlib import Path

# Top-level settings
QUALITY = "balanced"   # "balanced" | "max" | "debug"
MAX_N_REF = 48           # reduce for speed, increase for quality
TIMESTEP = 0             # static FLAME timestep to bake

NOTEBOOK_DIR = Path.cwd().resolve()
CONTAINERIZED = os.environ.get("CAP4D_CONTAINERIZED", "0") == "1"
CAP4D_DIR = Path(os.environ.get("CAP4D_PATH", NOTEBOOK_DIR.parent)).resolve()
RUNTIME_ROOT = Path(os.environ.get("CAP4D_RUNTIME_ROOT", CAP4D_DIR / ".runtime")).resolve()
PIXEL3DMM_DIR = Path(os.environ.get("PIXEL3DMM_PATH", RUNTIME_ROOT / "pixel3dmm")).resolve()

INPUT_VIDEO_PATH = str(Path(os.environ.get("CAP4D_INPUT_VIDEO_PATH", RUNTIME_ROOT / "my_head_video.mp4")).resolve())
OUTPUT_PATH = str((CAP4D_DIR / "examples/output/custom_static").resolve())
REPO_URL = "https://github.com/vikram-menon/cap4d.git"  # used only when CAP4D_DIR is not already a repo
REPO_REF = "colab"  # branch/tag/commit containing static export scripts when cloning is needed

print("CONTAINERIZED =", CONTAINERIZED)
print("CAP4D_DIR =", CAP4D_DIR)
print("RUNTIME_ROOT =", RUNTIME_ROOT)
print("PIXEL3DMM_DIR =", PIXEL3DMM_DIR)


In [ ]:
!nvidia-smi -L

In [ ]:
import shutil
import subprocess

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
repo_ready = (CAP4D_DIR / "requirements.txt").exists() and (CAP4D_DIR / "scripts" / "generate_static_avatar.sh").exists()

if repo_ready:
    print(f"Using existing CAP4D repo at {CAP4D_DIR}")
else:
    if CAP4D_DIR.exists():
        shutil.rmtree(CAP4D_DIR)
    subprocess.run(["git", "clone", REPO_URL, str(CAP4D_DIR)], check=True)
    subprocess.run(["git", "checkout", REPO_REF], cwd=str(CAP4D_DIR), check=True)
    print(f"Cloned CAP4D repo to {CAP4D_DIR}")


In [ ]:
import os
os.environ["CAP4D_PATH"] = str(CAP4D_DIR)
os.environ["PIXEL3DMM_PATH"] = str(PIXEL3DMM_DIR)
os.environ["PYTHONPATH"] = f"{CAP4D_DIR}:{os.environ.get('PYTHONPATH', '')}"
print('CAP4D_PATH=', os.environ['CAP4D_PATH'])
print('PIXEL3DMM_PATH=', os.environ['PIXEL3DMM_PATH'])


In [ ]:
%%bash
set -e
cd "$CAP4D_PATH"

if [[ "${CAP4D_CONTAINERIZED:-0}" == "1" ]]; then
  echo "Containerized environment detected; skipping base pip install cell."
else
  pip install -r requirements.txt
  export FORCE_CUDA=1
  pip install "git+https://github.com/facebookresearch/pytorch3d.git@stable"
fi

if ! command -v ffmpeg >/dev/null 2>&1; then
  echo "ffmpeg is required but not found on PATH. Please install ffmpeg locally and rerun this cell." >&2
  exit 1
fi


In [ ]:
import os
import getpass
os.environ['FLAME_USERNAME'] = input('FLAME username: ')
os.environ['FLAME_PWD'] = getpass.getpass('FLAME password: ')

In [ ]:
%%bash
set -e
cd "$CAP4D_PATH"
bash scripts/download_flame.sh
bash scripts/download_mmdm_weights.sh
bash scripts/install_pixel3dmm.sh


In [ ]:
from pathlib import Path
import py_compile
import re

base = PIXEL3DMM_DIR / 'scripts'

# 1) torch>=2.6 changed torch.load default to weights_only=True
p = base / 'network_inference.py'
s = p.read_text()

old = 'load_from_checkpoint(model_checkpoint, strict=False)'
new = 'load_from_checkpoint(model_checkpoint, strict=False, weights_only=False)'

if old in s:
    p.write_text(s.replace(old, new))
    print('Patched:', p, '(weights_only=False)')
else:
    print('Already patched or pattern not found:', p)

# 2) Prevent IndexError in facer segmentation when image_ids exceed local batch size

py_compile.compile(str(p), doraise=True)
print('Syntax OK')


In [ ]:
# Optional local file selection: use this cell if INPUT_VIDEO_PATH is not already available
from pathlib import Path

selected = ""
try:
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    selected = filedialog.askopenfilename(
        title='Select input video',
        filetypes=[('Video files', '*.mp4 *.mov *.avi *.mkv *.webm'), ('All files', '*.*')],
    ) or ""
    root.destroy()
except Exception as exc:
    print(f"GUI picker unavailable ({exc}).")

if selected:
    INPUT_VIDEO_PATH = str(Path(selected).expanduser().resolve())
else:
    manual = input('Enter local video path (leave blank to keep current INPUT_VIDEO_PATH): ').strip()
    if manual:
        INPUT_VIDEO_PATH = str(Path(manual).expanduser().resolve())

print('INPUT_VIDEO_PATH =', INPUT_VIDEO_PATH)


In [ ]:
import os
import re
import shlex
import subprocess
import threading
import time
from datetime import datetime
from pathlib import Path
LOG_DIR = str(RUNTIME_ROOT / "cap4d_logs")
os.makedirs(LOG_DIR, exist_ok=True)

# Rough ranges to give a practical expectation. These are not strict predictions.
ETA_HINTS_MIN = {
    "setup_clone": 1,
    "install_core": 8,
    "download_weights": 3,
    "install_pixel3dmm": 10,
    "tracking": 20,
    "generate_images": 30,
    "train_avatar": 45,
    "export_static": 1,
}
ETA_HINTS_MAX = {
    "setup_clone": 3,
    "install_core": 25,
    "download_weights": 10,
    "install_pixel3dmm": 35,
    "tracking": 90,
    "generate_images": 180,
    "train_avatar": 360,
    "export_static": 5,
}


def _now():
    return datetime.now().strftime("%H:%M:%S")


def run_logged(name, cmd, cwd=None, env=None, shell=False):
    """Run command with streamed logs, heartbeat, logfile, and rough ETA range."""
    log_path = Path(LOG_DIR) / f"{name}.log"
    start = time.time()
    last_line_time = [start]

    eta_min = ETA_HINTS_MIN.get(name)
    eta_max = ETA_HINTS_MAX.get(name)
    if eta_min is not None and eta_max is not None:
        print(f"[{_now()}] [{name}] ETA (rough): {eta_min}-{eta_max} min")

    print(f"[{_now()}] [{name}] START")
    print(f"[{_now()}] [{name}] CMD: {cmd if isinstance(cmd, str) else ' '.join(shlex.quote(c) for c in cmd)}")
    print(f"[{_now()}] [{name}] LOG: {log_path}")

    if shell:
        popen_cmd = cmd
    else:
        popen_cmd = cmd if isinstance(cmd, list) else shlex.split(cmd)

    process = subprocess.Popen(
        popen_cmd,
        cwd=cwd,
        env=env,
        shell=shell,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
    )

    stop_flag = {"stop": False}

    def heartbeat():
        while not stop_flag["stop"]:
            time.sleep(30)
            if stop_flag["stop"]:
                break
            elapsed = (time.time() - start) / 60.0
            silent = time.time() - last_line_time[0]
            msg = f"[{_now()}] [{name}] heartbeat: elapsed={elapsed:.1f}m"
            if silent >= 30:
                msg += f", no new output for {silent:.0f}s"
            if eta_max is not None:
                remaining_floor = max(0.0, eta_max - elapsed)
                msg += f", est remaining <= {remaining_floor:.1f}m"
            print(msg)

    t = threading.Thread(target=heartbeat, daemon=True)
    t.start()

    with log_path.open("w", encoding="utf-8") as f:
        for raw in process.stdout:
            line = raw.rstrip("\n")
            last_line_time[0] = time.time()
            elapsed = time.time() - start
            prefix = f"[{_now()}] [{name}] [+{elapsed:7.1f}s]"
            print(f"{prefix} {line}")
            f.write(raw)

    rc = process.wait()
    stop_flag["stop"] = True
    total = (time.time() - start) / 60.0

    if rc != 0:
        print(f"[{_now()}] [{name}] FAIL rc={rc} after {total:.1f}m")
        print(f"[{_now()}] [{name}] See log: {log_path}")
        raise RuntimeError(f"Step '{name}' failed (rc={rc}). Log: {log_path}")

    print(f"[{_now()}] [{name}] DONE in {total:.1f}m")
    return str(log_path)


print("Logging helper loaded. Logs will be written under:", LOG_DIR)


In [ ]:
# Run full static pipeline in one command with logs
import os
os.makedirs(OUTPUT_PATH, exist_ok=True)

run_logged(
    name="tracking",
    cmd=[
        "bash", "scripts/track_video_pixel3dmm.sh",
        INPUT_VIDEO_PATH,
        f"{OUTPUT_PATH}/reference_tracking",
        "--max_n_ref", str(MAX_N_REF),
    ],
    cwd=str(CAP4D_DIR),
)

run_logged(
    name="generate_images",
    cmd=[
        "python", "cap4d/inference/generate_images.py",
        "--config_path", "configs/generation/low_quality.yaml" if QUALITY == "balanced" else ("configs/generation/high_quality.yaml" if QUALITY == "max" else "configs/generation/debug.yaml"),
        "--reference_data_path", f"{OUTPUT_PATH}/reference_tracking",
        "--output_path", f"{OUTPUT_PATH}/mmdm",
    ],
    cwd=str(CAP4D_DIR),
)

run_logged(
    name="train_avatar",
    cmd=[
        "python", "gaussianavatars/train.py",
        "--config_path", "configs/avatar/low_quality.yaml" if QUALITY == "balanced" else ("configs/avatar/high_quality.yaml" if QUALITY == "max" else "configs/avatar/debug.yaml"),
        "--source_paths", f"{OUTPUT_PATH}/mmdm/reference_images/", f"{OUTPUT_PATH}/mmdm/generated_images/",
        "--model_path", f"{OUTPUT_PATH}/avatar/",
    ],
    cwd=str(CAP4D_DIR),
)

run_logged(
    name="export_static",
    cmd=[
        "python", "gaussianavatars/export_static_ply.py",
        "--model_path", f"{OUTPUT_PATH}/avatar/",
        "--source_paths", f"{OUTPUT_PATH}/mmdm/reference_images/", f"{OUTPUT_PATH}/mmdm/generated_images/",
        "--output_ply", f"{OUTPUT_PATH}/raw_static.ply",
        "--timestep", str(TIMESTEP),
    ],
    cwd=str(CAP4D_DIR),
)


In [ ]:
from pathlib import Path

# Patch cap4d/datasets/utils.py to handle missing .mp4 extension
p = CAP4D_DIR / "cap4d" / "datasets" / "utils.py"
content = p.read_text()

old_code = """def load_frame(
    video_path: Path,  # path to .mp4 or dir containing frames
    frame_id: np.ndarray,
):
    if (video_path).is_dir():"""

new_code = """def load_frame(
    video_path: Path,  # path to .mp4 or dir containing frames
    frame_id: np.ndarray,
):
    # PATCH: Check for .mp4 if path doesn't exist
    if not video_path.exists() and video_path.with_suffix('.mp4').exists():
        video_path = video_path.with_suffix('.mp4')

    if (video_path).is_dir():"""

if old_code in content:
    new_content = content.replace(old_code, new_code)
    p.write_text(new_content)
    print("Patched cap4d/datasets/utils.py successfully.")
else:
    print("Patch pattern not found. File might vary or already be patched.")
    # Check if already patched
    if "if not video_path.exists()" in content:
        print("File appears to be already patched.")


In [ ]:
# Resume pipeline from generate_images

run_logged(
    name="generate_images",
    cmd=[
        "python", "cap4d/inference/generate_images.py",
        "--config_path", "configs/generation/low_quality.yaml" if QUALITY == "balanced" else ("configs/generation/high_quality.yaml" if QUALITY == "max" else "configs/generation/debug.yaml"),
        "--reference_data_path", f"{OUTPUT_PATH}/reference_tracking",
        "--output_path", f"{OUTPUT_PATH}/mmdm",
    ],
    cwd=str(CAP4D_DIR),
)

run_logged(
    name="train_avatar",
    cmd=[
        "python", "gaussianavatars/train.py",
        "--config_path", "configs/avatar/low_quality.yaml" if QUALITY == "balanced" else ("configs/avatar/high_quality.yaml" if QUALITY == "max" else "configs/avatar/debug.yaml"),
        "--source_paths", f"{OUTPUT_PATH}/mmdm/reference_images/", f"{OUTPUT_PATH}/mmdm/generated_images/",
        "--model_path", f"{OUTPUT_PATH}/avatar/",
    ],
    cwd=str(CAP4D_DIR),
)

run_logged(
    name="export_static",
    cmd=[
        "python", "gaussianavatars/export_static_ply.py",
        "--model_path", f"{OUTPUT_PATH}/avatar/",
        "--source_paths", f"{OUTPUT_PATH}/mmdm/reference_images/", f"{OUTPUT_PATH}/mmdm/generated_images/",
        "--output_ply", f"{OUTPUT_PATH}/raw_static.ply",
        "--timestep", str(TIMESTEP),
    ],
    cwd=str(CAP4D_DIR),
)

In [ ]:
import os

file_path = str(CAP4D_DIR / "scripts" / "pixel3dmm" / "l2cs_eye_tracker.py")

with open(file_path, "r") as f:
    lines = f.readlines()

new_lines = []
patched = False
for line in lines:
    # Locate the line with the problematic argument and comment it out
    if "face_detector_kwargs=" in line and not line.strip().startswith("#"):
        print(f"Patching line: {line.strip()}")
        new_lines.append(f"            # {line.strip()}  # Patched: argument removed\n")
        patched = True
    else:
        new_lines.append(line)

if patched:
    with open(file_path, "w") as f:
        f.writelines(new_lines)
    print("Successfully patched l2cs_eye_tracker.py")
    print("You can now rerun the tracking step.")
else:
    print("Pattern not found. The file might already be patched or the code structure is different.")


In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-build-isolation",
        "git+https://github.com/NVlabs/nvdiffrast.git",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-c", "import nvdiffrast.torch as dr; print('nvdiffrast OK')"],
    check=True,
)


In [ ]:
        # for batch_idx in range(ceil(len(image_stack)/batch_size)):
        #     image_batch = torch.cat(image_stack[batch_idx*batch_size:(batch_idx+1)*batch_size], dim=0)
        #     frame_idx_batch = frame_stack[batch_idx*batch_size:(batch_idx+1)*batch_size]
        #     og_shape_batch = original_shapes[batch_idx*batch_size:(batch_idx+1)*batch_size]

        #     #if True:
        #     try:
        #         with torch.inference_mode():
        #             faces = face_detector(image_batch)
        #             torch.cuda.empty_cache()
        #             faces = face_parser(image_batch, faces, bbox_scale_factor=1.25)
        #             torch.cuda.empty_cache()

        #         seg_logits = faces['seg']['logits']
        #         back_ground = torch.all(seg_logits == 0, dim=1, keepdim=True).detach().squeeze(1).cpu().numpy()
        #         seg_probs = seg_logits.softmax(dim=1)  # nfaces x nclasses x h x w
        #         seg_classes = seg_probs.argmax(dim=1).detach().cpu().numpy().astype(np.uint8)
        #         seg_classes[back_ground] = seg_probs.shape[1] + 1

        #         for _iidx in range(seg_probs.shape[0]):
        #             idx = int(_iidx)
        #             if idx < 0 or idx >= len(frame_idx_batch):
        #                 continue
        #             frame = frame_idx_batch[idx]
        #             iidx = faces['image_ids'][_iidx].item()
        #             try:
        #                 I_color = viz_results(
        #                     image_batch[iidx:iidx+1],
        #                     seq_classes=seg_classes[_iidx:_iidx+1],
        #                     n_classes=seg_probs.shape[1] + 1,
        #                     suppress_plot=True
        #                 )
        #                 I_color.save(f'{out_seg_annot}/color_{frame}.png')
        #             except Exception as ex:
        #                 pass
        #             I = Image.fromarray(seg_classes[_iidx])
        #             I.save(f'{out_seg}/{frame}.png')
        #         torch.cuda.empty_cache()
        #     except Exception as exx:
        #         traceback.print_exc()
        #         continue

In [ ]:
from pathlib import Path
import shutil
from IPython.display import FileLink, display

ply_path = Path(f"{OUTPUT_PATH}/raw_static.ply").expanduser().resolve()
print('Exported PLY:', ply_path)

if ply_path.exists():
    display(FileLink(str(ply_path)))

    copied = False
    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        destination = filedialog.asksaveasfilename(
            title='Optional: Save a copy of raw_static.ply',
            initialfile=ply_path.name,
            defaultextension='.ply',
            filetypes=[('PLY files', '*.ply'), ('All files', '*.*')],
        )
        root.destroy()
        if destination:
            dst = Path(destination).expanduser().resolve()
            shutil.copy2(ply_path, dst)
            print('Copied to:', dst)
            copied = True
    except Exception as exc:
        print(f"GUI save dialog unavailable ({exc}).")

    if not copied:
        manual_destination = input('Optional copy destination path (leave blank to skip): ').strip()
        if manual_destination:
            dst = Path(manual_destination).expanduser().resolve()
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(ply_path, dst)
            print('Copied to:', dst)
else:
    print('PLY file not found yet. Run export_static first.')
